# Final demo — visualizations and `predict_sentiment()`

- **Sentiment distribution** — pie chart from the processed dataset (**negative / neutral / positive** labels)
- **Word clouds** — one cloud each for **positive**, **negative**, and **neutral** (`text_clean`)
- **Predictions** — `predict_sentiment()` uses the **multiclass** model (0/1/2) trained on the same labels
- **Artifacts** — `models/tfidf_vectorizer.pkl` + `models/best_model.pkl`

**Prerequisites:** run `preprocessing.ipynb`, `feature_extraction.ipynb`, and `model_training.ipynb` after regenerating `sample_dataset.csv` with `sentiment_analysis.ipynb`.

In [ ]:
%pip install -q pandas matplotlib wordcloud joblib scikit-learn nltk

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wordcloud import WordCloud

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from predict_sentiment import predict_sentiment

DATA_DIR = ROOT / "data"
PROCESSED_CSV = DATA_DIR / "processed_dataset.csv"

PIE_PATH = NOTEBOOK_DIR / "sentiment_distribution_pie.png"
WC_COMBO_PATH = NOTEBOOK_DIR / "wordclouds_pos_neg_neu.png"

In [ ]:
df = pd.read_csv(PROCESSED_CSV)
counts = df["sentiment"].value_counts()
pie_palette = {"negative": "#c44e52", "neutral": "#888888", "positive": "#4c72b0"}
pie_colors = [pie_palette.get(str(l).lower(), "#999999") for l in counts.index]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    counts.values,
    labels=counts.index,
    autopct="%1.1f%%",
    colors=pie_colors,
    startangle=90,
)
ax.set_title("Sentiment distribution (processed dataset)")
plt.tight_layout()
plt.savefig(PIE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", PIE_PATH)

In [ ]:
def corpus_for_cloud(mask: pd.Series) -> str:
    parts = df.loc[mask, "text_clean"].astype(str).str.strip()
    parts = parts[parts.str.len() > 0]
    return " ".join(parts.tolist())


pos_text = corpus_for_cloud(df["sentiment"].str.lower() == "positive")
neg_text = corpus_for_cloud(df["sentiment"].str.lower() == "negative")
neu_text = corpus_for_cloud(df["sentiment"].str.lower() == "neutral")

wc_common = dict(width=700, height=450, background_color="white", max_words=120, collocations=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, title, txt, cmap in zip(
    axes,
    ["positive", "negative", "neutral"],
    [pos_text, neg_text, neu_text],
    ["Blues", "Reds", "Greys"],
):
    if txt:
        wc = WordCloud(**wc_common, colormap=cmap).generate(txt)
        ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Word cloud — {title}")
    ax.axis("off")

plt.tight_layout()
plt.savefig(WC_COMBO_PATH, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", WC_COMBO_PATH)

In [ ]:
# Model output distribution on a random sample (multiclass: neg / neutral / pos)
sample_n = min(300, len(df))
sample_df = df.sample(sample_n, random_state=42)
preds = [predict_sentiment(t) for t in sample_df["text"].astype(str)]
pred_counts = pd.Series(preds).value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
color_map = {
    "positive": "#4c72b0",
    "negative": "#c44e52",
    "neutral": "#888888",
    "unknown": "#bbbbbb",
}
colors = [color_map.get(x, "#999999") for x in pred_counts.index]
ax.pie(
    pred_counts.values,
    labels=pred_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
)
ax.set_title(f"predict_sentiment() outputs (n={sample_n})")
plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / "prediction_distribution_pie.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", NOTEBOOK_DIR / "prediction_distribution_pie.png")

In [ ]:
examples = [
    "This product exceeded all my expectations! https://t.co/abc",
    "Terrible service, would not recommend to anyone.",
    "Not bad at all — pleasantly surprised.",
    "The meeting is scheduled for 3pm tomorrow.",
    "ok thanks",
    "I suppose it could have been worse.",
]

rows = [{"text": t, "predict_sentiment": predict_sentiment(t)} for t in examples]
display(pd.DataFrame(rows))